# HE SDK playground

A minimal trusted-local notebook for learning homomorphic encryption through `he_looming_sdk`. It creates one OpenFHE session, reuses its keys, encrypts small vectors, executes an HE function, decrypts the result, and checks the CKKS approximation.

The notebook code itself does **not** call PostgreSQL, HTTP services, or GPU infrastructure. In the recommended server deployment, CI packages Python, the SDK wheel, OpenFHE, and JupyterLab into one CPU image.

## Setup

The deployed JupyterLab image is already prepared; start at **Steps** below. Only for a local installation, create the environment from a terminal before opening this notebook:

```sh
python3 -m venv .venv-he-notebook
. .venv-he-notebook/bin/activate
python -m pip install --upgrade pip
python -m pip install jupyterlab openfhe==1.5.1.0.24.4
python -m pip install he_looming_sdk==0.3.1
python -m jupyter lab notebooks/he_playground.ipynb
```

The distribution is named `he_looming_sdk`, while Python imports remain under `he_sdk`.

## Steps

### 1. Import the SDK and choose small inputs

In [ ]:
import atexit
import json
import math
from time import perf_counter

from he_sdk import HESession

LEFT = [1.25, -2.0, 3.5, 4.0]
RIGHT = [0.75, 5.0, -1.5, 2.0]
ABSOLUTE_TOLERANCE = 1e-3
RELATIVE_TOLERANCE = 1e-3

### 2. Create one reusable HE session

OpenFHE context and key generation are the expensive startup work. Keep this session alive while experimenting instead of creating a session for every operation. Re-running this cell safely closes the previous session first.

In [ ]:
if "he" in globals():
    he.close()

setup_started = perf_counter()
he = HESession.create(backend="openfhe")
atexit.register(he.close)

print("backend:", he.capabilities.backend)
print("operations:", he.capabilities.operations)
print("profile fingerprint:", he.config.fingerprint)
print("session setup seconds:", round(perf_counter() - setup_started, 3))

### 3. See the direct encrypt → square → decrypt path

In [ ]:
operation_started = perf_counter()
encrypted = he.encrypt([2.0, 3.0])
encrypted_squared = he.square(encrypted)
decrypted = he.decrypt(encrypted_squared)
expected = [4.0, 9.0]

assert all(
    math.isclose(actual, wanted, rel_tol=RELATIVE_TOLERANCE, abs_tol=ABSOLUTE_TOLERANCE)
    for actual, wanted in zip(decrypted, expected, strict=True)
)

print("decrypted:", decrypted)
print("expected:", expected)
print("operation seconds:", round(perf_counter() - operation_started, 3))
print("MINIMAL_HE_SMOKE=PASS")

### 4. Add a small notebook-friendly operation helper

The helper still uses the real SDK objects. It only removes repetitive notebook code and compares decrypted CKKS output with normal Python arithmetic.

In [ ]:
BINARY_OPERATIONS = {"add", "subtract", "multiply"}


def expected_result(operation, left, right=None):
    if operation == "add":
        return [a + b for a, b in zip(left, right, strict=True)]
    if operation == "subtract":
        return [a - b for a, b in zip(left, right, strict=True)]
    if operation == "multiply":
        return [a * b for a, b in zip(left, right, strict=True)]
    if operation == "square":
        return [value * value for value in left]
    if operation == "sum":
        return sum(left)
    if operation == "mean":
        return sum(left) / len(left)
    if operation == "variance":
        mean = sum(left) / len(left)
        return sum((value - mean) ** 2 for value in left) / len(left)
    raise ValueError(f"unsupported operation: {operation}")


def run_he(operation, left, right=None):
    if operation not in he.capabilities.operations:
        raise ValueError(f"backend does not support {operation!r}")
    if operation in BINARY_OPERATIONS and right is None:
        raise ValueError(f"{operation} requires a right-hand vector")

    started = perf_counter()
    encrypted_left = he.encrypt(left)
    if operation in BINARY_OPERATIONS:
        encrypted_right = he.encrypt(right)
        encrypted_result = getattr(he, operation)(encrypted_left, encrypted_right)
    else:
        encrypted_result = getattr(he, operation)(encrypted_left)

    observed = he.decrypt(encrypted_result)
    expected = expected_result(operation, left, right)
    observed_values = observed if isinstance(observed, list) else [observed]
    expected_values = expected if isinstance(expected, list) else [expected]
    errors = [
        abs(actual - wanted)
        for actual, wanted in zip(observed_values, expected_values, strict=True)
    ]
    passed = all(
        math.isclose(actual, wanted, rel_tol=RELATIVE_TOLERANCE, abs_tol=ABSOLUTE_TOLERANCE)
        for actual, wanted in zip(observed_values, expected_values, strict=True)
    )
    return {
        "operation": operation,
        "observed": observed,
        "expected": expected,
        "maximum_absolute_error": max(errors),
        "seconds": round(perf_counter() - started, 6),
        "status": "PASS" if passed else "FAIL",
    }

### 5. Choose an operation and play

Change `OPERATION` to `add`, `subtract`, `multiply`, `square`, `sum`, `mean`, or `variance`. `RIGHT` is used only for the three binary operations.

In [ ]:
OPERATION = "variance"

result = run_he(OPERATION, LEFT, RIGHT)
print(json.dumps(result, indent=2, sort_keys=True))
assert result["status"] == "PASS"

## Checks

Set `RUN_ALL` to `True` only when you want the complete seven-operation check. The same HE session is reused.

In [ ]:
RUN_ALL = False

if RUN_ALL:
    suite = [
        run_he("add", LEFT, RIGHT),
        run_he("subtract", LEFT, RIGHT),
        run_he("multiply", LEFT, RIGHT),
        run_he("square", LEFT),
        run_he("sum", LEFT),
        run_he("mean", LEFT),
        run_he("variance", LEFT),
    ]
    print(json.dumps(suite, indent=2, sort_keys=True))
    assert all(item["status"] == "PASS" for item in suite)
else:
    print("Full suite skipped; set RUN_ALL=True when needed.")

## Next steps

Keep the notebook code simple until the HE behavior is understood. PostgreSQL can later store experiment metadata or encrypted artifacts, but it should not be required to create a session or run an operation. FIDES/GPU and HTTP execution remain separate follow-up paths.

When finished, run `he.close()` or shut down the notebook kernel.